# Исследование информации о компьютерных играх в 2000-2013гг.

- Автор: Севрюкова В.Ю.
- Дата:

### Цели и задачи проекта
 В этой тетрадке используем датасет /datasets/new_games.csv, который содержит информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр.
 
 Задача — познакомиться с данными, проверить их корректность и провести предобработку, получив необходимый срез данных.
 
  В статье-исследовании хотят сделать обзор игровых платформ, изучить объёмы продаж игр разных жанров и региональные предпочтения игроков. Акцент хотят сделать на играх жанра RPG — так называют компьютерные ролевые игры, в которых игроки управляют персонажами.

### Описание данных
Данные /datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
* Name — название игры.
* Platform — название платформы.
* Year of Release — год выпуска игры.
* Genre — жанр игры.
* NA sales — продажи в Северной Америке (в миллионах проданных копий).
* EU sales — продажи в Европе (в миллионах проданных копий).
* JP sales — продажи в Японии (в миллионах проданных копий).
* Other sales — продажи в других странах (в миллионах проданных копий).
* Critic Score — оценка критиков (от 0 до 100).
* User Score — оценка пользователей (от 0 до 10).
* Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.


### Содержимое проекта

1. Отбор данных по времени выхода игры.
2. Категоризация игры по оценкам пользователей и экспертов.
3. Определение топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

---

## 1. Загрузка данных и знакомство с ними


In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм games
games = pd.read_csv('https://code.s3.yandex.net//datasets/new_games.csv')

In [3]:
display(games.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


None

In [4]:
display(games.head())

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет /datasets/new_games.csv содержит 11 столбцов и 16956 строк, в которых представлена информация о компьютерных играх, объем данных составляет 1.4+МБ.


Изучим типы данных и их корректность:

- **Числовые значения с плавающей запятой (float64).**Четыре столбца:`Year of Release`(содержит год выпуска игры),'NA sales'(содержит информацию о продажах в Северной Америке (в миллионах проданных копий)),'Other sales'(содержит информацию о продажах в других странах (в миллионах проданных копий)),'Critic Score'(содержит оценка критиков (от 0 до 100)), и представлены типом `float64`. 


-  **Строковые данные (object).** Семь столбцов имеют тип данных `object`: `Name`(содержит информацию о названии игры), 'Platform '(содержит информацию о названии платформы),'Genre'(содержит информацию о жанре игры),'EU sales'(содержит информацию о продажах в Европе (в миллионах проданных копий)),'JP sales'(содержит информацию о продажах в Японии (в миллионах проданных копий)),'User Score'(содержит информацию об  оценках пользователей (от 0 до 10)),'Rating'(содержит информацию о рейтинге организации ESRB)-тип данных `object`.


Пропуски встречаются в шести столбцах, за исключением 'Platform','NA sales','EU sales','JP sales','Other sales'.   
После анализа типов данных видно, что часть столбцов, представленных типом `float64`, не совсем корректно, учитывая названия и содержание столбцов. Для оптимизации можно использовать целочисленные типы с уменьшенной разрядностью в тип `datetime64`.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [5]:
# Выводим названия всех столбцов
print("Названия столбцов датафрейма:")
print(games.columns.tolist())

Названия столбцов датафрейма:
['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


In [6]:
# Смотрим типы данных
games.dtypes

Name                object
Platform            object
Year of Release    float64
Genre               object
NA sales           float64
EU sales            object
JP sales            object
Other sales        float64
Critic Score       float64
User Score          object
Rating              object
dtype: object

In [7]:
# Приводим названия столбцов к змеиному стилю
games.columns = games.columns.str.lower().str.replace(' ', '_')
games.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

In [8]:
display(games)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16951,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.0,0.00,NaN,NaN,NaN
16953,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16954,Spirits & Spells,GBA,2003.0,Platform,0.01,0.0,0.0,0.00,NaN,NaN,NaN


### 2.2. Типы данных

In [9]:
# Опередлим тип данных всех столбцов
games.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [10]:
# Определим уникальные значения во всех столбцах
for column in games.columns:
    print(f"Столбец: {column}")
    print(f"Уникальные значения: {games[column].unique()}")
    print("-" * 40)

Столбец: name
Уникальные значения: ['Wii Sports' 'Super Mario Bros.' 'Mario Kart Wii' ...
 'Woody Woodpecker in Crazy Castle 5' 'LMA Manager 2007'
 'Haitaka no Psychedelica']
----------------------------------------
Столбец: platform
Уникальные значения: ['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
----------------------------------------
Столбец: year_of_release
Уникальные значения: [2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015. 2012. 2014. 1992. 1997.
 1993. 1994. 1982. 2016. 2003. 1986. 2000.   nan 1995. 1991. 1981. 1987.
 1980. 1983.]
----------------------------------------
Столбец: genre
Уникальные значения: ['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' nan 'MISC'
 'ROLE-PLAYING' 'RACIN

Возможные причины некорртектных типов данных:
- Наличие пропусков (NaN, пустые строки, специальные метки).
Значения типа 'N/A', 'NULL', '' (пустая строка) не распознаются как числа. При преобразовании они становятся NaN.
- Строковые значения в числовых столбцах. 
В числовом столбце могут быть значения вроде '1 000', '1,234.56', '1000 USD'. Запятые, пробелы, валюта мешают преобразованию в число.
- Смешанные типы в одном столбце.
Например, часть строк — числа, часть — текст ('123', 'error', '456'). Pandas сохраняет весь столбец как object (строковый тип).
- Нестандартные разделители.
В некоторых данных десятичный разделитель — запятая (,), а не точка (.), либо используются разные форматы чисел.
- Опечатки и мусорные данные.
Случайно попавшие символы ('123abc', 'nan', 'NULL'), лишние пробелы.

In [11]:
# Опередлим тип данных всех столбцов
games.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [12]:
# Перечень столбцов, которые должны быть числовыми
numeric_columns = ['year_of_release','na_sales','eu_sales','jp_sales','other_sales','critic_score','user_score']

In [13]:
# Преобразование типов данных на числовые и замена всех значений на NaN, которые не удалось преобразовать в число
for col in numeric_columns:
    games[col] = pd.to_numeric(games[col],errors='coerce')      

In [14]:
games.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

### 2.3. Наличие пропусков в данных

In [15]:
# Находим количество пропусков в каждом столбце в абсолютном значении
count = games.isna().sum()
count

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [16]:
# Подсчёт процента пропусков
count_= (games.isna().sum() / len(games)) * 100
count_

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

Вывод: большое количество пропусков встречается в столбцах 'Critic Score'(51%),'User Score'(55%),'Rating'(41%).
Пропуски в данных могут возникать по разным причинам: 
- данные не были введены или зарегистрированы; 
- ошибки при сборе или передаче данных; 
- несовместимость форматов данных при объединении различных источников; 
- преднамеренное удаление данных из-за их недостоверности или неактуальности. 

Пропуски можно пропустить, удалить или самостоятельно запонить.

In [17]:
# Замена пропусков в столбце 'NA sales' на среднее значение
games['na_sales'] = games['na_sales'].fillna(games.groupby(['platform','year_of_release'])['na_sales'].transform('mean'))

In [18]:
display(games['na_sales'])

0        41.36
1        29.08
2        15.68
3        15.61
4        11.27
         ...  
16951     0.00
16952     0.00
16953     0.00
16954     0.01
16955     0.00
Name: na_sales, Length: 16956, dtype: float64

In [19]:
# Замена пропусков в столбце 'EU sales' на среднее значение
games['eu_sales'] = games['eu_sales'].fillna(games.groupby(['platform','year_of_release'])['eu_sales'].transform('mean'))

In [20]:
display(games['eu_sales'])

0        28.96
1         3.58
2        12.76
3        10.93
4         8.89
         ...  
16951     0.00
16952     0.01
16953     0.00
16954     0.00
16955     0.00
Name: eu_sales, Length: 16956, dtype: float64

In [21]:
# Замена пропусков в столбце 'JP sales' на среднее значение
games['jp_sales'] = games['jp_sales'].fillna(games.groupby(['platform', 'year_of_release'])['jp_sales'].transform('mean'))

In [22]:
display(games['jp_sales'])

0         3.77
1         6.81
2         3.79
3         3.28
4        10.22
         ...  
16951     0.01
16952     0.00
16953     0.01
16954     0.00
16955     0.01
Name: jp_sales, Length: 16956, dtype: float64

In [23]:
print(f"До удаления: {len(games)} строк")
print(f"Строк с пропусками: {games.isna().any(axis=1).sum()}")

До удаления: 16956 строк
Строк с пропусками: 10045


In [24]:
# Удаление пропущенных значений в столбе
games_clean = games.dropna(subset=['name','year_of_release','genre'])

In [25]:
print(f"После удаления: {len(games_clean)} строк")
print(f"Удалено строк: {len(games) - len(games_clean)}") 

После удаления: 16679 строк
Удалено строк: 277


In [26]:
display(games)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16951,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.00,0.00,NaN,NaN,NaN
16953,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16954,Spirits & Spells,GBA,2003.0,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN


### 2.4. Явные и неявные дубликаты в данных

In [27]:
# Изучим уникальные значения с названиями жанра игры, платформы, рейтинга и года выпуска
games['genre'].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan, 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION',
       'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE',
       'SIMULATION', 'PUZZLE', 'STRATEGY'], dtype=object)

In [28]:
display(games['platform'].unique())

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [29]:
display(games['name'].unique())

array(['Wii Sports', 'Super Mario Bros.', 'Mario Kart Wii', ...,
       'Woody Woodpecker in Crazy Castle 5', 'LMA Manager 2007',
       'Haitaka no Psychedelica'], dtype=object)

In [30]:
display(games['year_of_release'].unique()) 

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [31]:
# Проведем нормализацию данных, сначала приведем к нижнему регистру поля 'Genre','Name'
games['genre'].str.lower()

0              sports
1            platform
2              racing
3              sports
4        role-playing
             ...     
16951          action
16952          sports
16953       adventure
16954        platform
16955      simulation
Name: genre, Length: 16956, dtype: object

In [32]:
 games['name'].str.lower()

0                           wii sports
1                    super mario bros.
2                       mario kart wii
3                    wii sports resort
4             pokemon red/pokemon blue
                     ...              
16951    samurai warriors: sanada maru
16952                 lma manager 2007
16953          haitaka no psychedelica
16954                 spirits & spells
16955              winning post 8 2016
Name: name, Length: 16956, dtype: object

In [33]:
# Приведем к верхнему регистру поле 'Rating'
games['rating'].str.upper()

0          E
1        NaN
2          E
3          E
4        NaN
        ... 
16951    NaN
16952    NaN
16953    NaN
16954    NaN
16955    NaN
Name: rating, Length: 16956, dtype: object

In [34]:
# Проверим наличие явных дубликатов в данных
duplicates = games[games.duplicated()]

In [35]:
# Количество дубликатов
duplicates.shape[0] 

182

In [36]:
# Удалим дубликаты 
games_clean = games.drop_duplicates()  

In [37]:
# Сохраняем исходную длину датафрейма
original_length = len(games)
# Считаем количество удалённых строк
removed_duplicates = original_length - len(games_clean)

print(f"Удалено дубликатов: {removed_duplicates}")

Удалено дубликатов: 182


Обнаружено 182 строки явных дубликатов. Дубликаты можно пропустить, или удалить, или заполнить (к примеру, средним значением по столбцу).

In [38]:
# Определим исходный датафрейм и его количество строк
original_df = games

In [39]:
original_count = len(original_df)

In [40]:
display(original_count)

16956

In [41]:
# Определим датафрейм, используемый после удаления дубликатов, и его количество строк
cleaned_df=original_df.drop_duplicates() 

In [42]:
cleaned_count = len(cleaned_df)

In [43]:
display(cleaned_count)

16774

In [44]:
# Абсолютное количество удалённых строк
deleted_count = original_count - cleaned_count

In [45]:
display(deleted_count)

182

In [46]:
# Относительное количество удалённых строк (%)
deleted=(deleted_count/original_count-1)*100

In [47]:
display(deleted)

-98.92663364000943

В данном пункте мы определили уникальные значения в столбцах: 'Genre', 'Platform', 'Rating', 'Year of Release', и провели нормализацию в них. 
Определили явные дубликаты (их количество составило 182 строки), количество строк исходного датафрейма - 16956, количество после удаления дубликатов - 16774. В относительном значении количество удаленных строк составляет 1.073%. 

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. 

In [48]:
# Сохраним новый срез данных в отдельном датафрейме
df_actual = games.loc[(games['year_of_release']>=2000)&(games['year_of_release']<=2013)].copy()

In [49]:
display(df_actual)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
6,New Super Mario Bros.,DS,2006.0,Platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,Misc,13.96,9.18,2.93,2.84,58.0,6.6,E
...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003.0,Shooter,0.01,0.00,0.00,0.00,NaN,NaN,T
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002.0,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN
16950,SCORE International Baja 1000: The Official Game,PS2,2008.0,Racing,0.00,0.00,0.00,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.00,0.00,NaN,NaN,NaN


---

## 4. Категоризация данных

In [50]:
# Создадим копию и разделим все игры по оценкам пользователей и выделим категории
df_actual = df_actual.copy()

In [51]:
def score_category(score):
    if score >= 8:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

In [52]:
df_actual.loc[:,'user_score_group']= df_actual['user_score'].apply(score_category) 

In [53]:
# Мы создали новый столбец 'user_score_group', который будет содержать 3 категориии
display(df_actual['user_score_group'])

0        высокая оценка
2        высокая оценка
3        высокая оценка
6        высокая оценка
7        средняя оценка
              ...      
16947     низкая оценка
16949     низкая оценка
16950     низкая оценка
16952     низкая оценка
16954     низкая оценка
Name: user_score_group, Length: 12980, dtype: object

In [54]:
# Разделим все игры по оценкам критиков и выделим категории
def score_category(score):
    if score >= 80:
        return 'высокая оценка'
    elif score >= 30:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

In [55]:
df_actual.loc[:,'user_score_group_2']= df_actual['critic_score'].apply(score_category) 

In [56]:
# Мы создали новый столбец 'user_score_group_2', который будет содержать 3 категориии.
display(df_actual['user_score_group_2'])

0        средняя оценка
2        высокая оценка
3        высокая оценка
6        высокая оценка
7        средняя оценка
              ...      
16947     низкая оценка
16949     низкая оценка
16950     низкая оценка
16952     низкая оценка
16954     низкая оценка
Name: user_score_group_2, Length: 12980, dtype: object

In [57]:
# Сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории
category_counts = df_actual['user_score_group'].value_counts(dropna=False)

In [58]:
display(category_counts)

низкая оценка     6525
средняя оценка    4148
высокая оценка    2307
Name: user_score_group, dtype: int64

In [59]:
category_counts_2 = df_actual['user_score_group_2'].value_counts(dropna=False)

In [60]:
display(category_counts)

низкая оценка     6525
средняя оценка    4148
высокая оценка    2307
Name: user_score_group, dtype: int64

In [61]:
# Определим Топ-7 платформ по количеству игр, выпущенные за 2000-2013 гг.
top_7 = df_actual.groupby('platform').size().sort_values(ascending=False).head(7)

In [62]:
display(top_7)

platform
PS2     2154
DS      2146
Wii     1294
PSP     1199
X360    1138
PS3     1107
GBA      826
dtype: int64

---

## 5. Итоговый вывод

Были загружены данные /datasets/new_games.csv содержит 11 столбцов и 16956 строк, в которых представлена информация о компьютерных играх, объем данных составляет 1.4+МБ. Изучим типы данных и их корректность:

Числовые значения с плавающей запятой (float64).Четыре столбца:Year of Release(содержит год выпуска игры),'NA sales'(содержит информацию о продажах в Северной Америке (в миллионах проданных копий)),'Other sales'(содержит информацию о продажах в других странах (в миллионах проданных копий)),'Critic Score'(содержит оценка критиков (от 0 до 100)), и представлены типом float64.
Строковые данные (object). Семь столбцов имеют тип данных object: Name(содержит информацию о названии игры), 'Platform '(содержит информацию о названии платформы),'Genre'(содержит информацию о жанре игры),'EU sales'(содержит информацию о продажах в Европе (в миллионах проданных копий)),'JP sales'(содержит информацию о продажах в Японии (в миллионах проданных копий)),'User Score'(содержит информацию об оценках пользователей (от 0 до 10)),'Rating'(содержит информацию о рейтинге организации ESRB)-тип данных object.

Пропуски встречаются в шести столбцах, за исключением 'Platform','NA sales','EU sales','JP sales','Other sales'.
После анализа типов данных видно, что часть столбцов, представленных типом float64, не совсем корректно, учитывая названия и содержание столбцов. Для оптимизации можно использовать целочисленные типы с уменьшенной разрядностью в тип datetime64.

Вывод:
большое количество пропусков встречается в столбцах 'Critic Score'(51%),'User Score'(55%),'Rating'(41%). 
Для оптимизации работы с данными в датафрейме были произведены следующие изменения типов данных:
столбцы 'Year of Release','NA sales','EU sales','JP sales','Other sales','Critic Score','User Score'переведенычисловой тип данных.
Для дополнительной работы с данными были созданы дополнительные столбцы:
'count' - количество пропусков в каждом столбце в абсолютном значении,'count_'- % пропусков, 'duplicates'- список явных дубликатов, 'original_count'- количество строк, 'deleted_count'- количество удаленных строк, 'category'- категории по оценкам пользователей.
​
​
​
​